In [5]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, ToolMessage, HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
import os
from pathlib import Path
from dotenv import load_dotenv

In [6]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [8]:
@tool
def add(a: int, b: int):
    """Adds two numbers."""

    return a + b

tools = [add]

model = ChatGroq(model="llama-3.1-8b-instant").bind_tools(tools)



In [11]:
def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are a helpful assistant. You can use the tools provided to answer questions.")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

In [12]:
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]

    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"
    

In [19]:
graph = StateGraph(AgentState)
graph.add_node("Agent", model_call)

tool_node = ToolNode(tools=tools)
graph.add_node("Tool", tool_node)

graph.add_edge(START, "Agent")
graph.add_conditional_edges(
    "Agent",
    should_continue,
    {
        "continue": "Tool",
        "end": END
    }
)
graph.add_edge("Tool", "Agent")


app = graph.compile()

def print_stream(stream):
    for chunk in stream:
        message = chunk["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [("user", "What is 2 + 2?")]} 
print_stream(app.stream(inputs, stream_mode ="values"))

================================ Human Message =================================

What is 2 + 2?
================================== Ai Message ==================================
Tool Calls:
  add (drpa5nnkz)
 Call ID: drpa5nnkz
  Args:
    a: 2
    b: 2
================================= Tool Message =================================
Name: add

4
================================== Ai Message ==================================

The result of the function call is 4.
